# 07 · Set operations: UNION, INTERSECT, EXCEPT (BigQuery)

Set operations stack two query results **on top of each other** (joins put them side by side).
Both queries need the same number of columns, in the same order, with matching types.

| | BigQuery | Oracle | Spark |
|---|---|---|---|
| all rows | `UNION ALL` | `UNION ALL` | `UNION ALL` |
| no duplicates | `UNION DISTINCT` | `UNION` | `UNION` |
| in both | `INTERSECT DISTINCT` | `INTERSECT` | `INTERSECT` |
| in first, not second | `EXCEPT DISTINCT` | `MINUS` (23ai also `EXCEPT`) | `EXCEPT` |

**How to run:** Start each SQL cell with `%%bigquery`. Write tables as `shield.table_name`.
Each cell prints how much data it read. `%%bigquery dry` checks the size for free.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`bigquery/practice/07_set_operations.ipynb`](../../bigquery/practice/07_set_operations.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "bigquery"))
%load_ext bq_helper

## Q1. One list of people  ·  🟢 easy

Make one list of **all** employees and customers: `person_type` (`employee` / `customer`), `person_id`, `name`.

*Expected result: 70 rows · columns: `person_type`, `person_id`, `name`*

**How it works:** Two SELECTs with the same column shape, stacked with `UNION ALL`. A fixed text column (`'employee'`) tells where each row came from.

In [ ]:
%%bigquery
SELECT 'employee' AS person_type, emp_id AS person_id, name FROM shield.employees
UNION ALL
SELECT 'customer', customer_id, name FROM shield.customers

,person_type,person_id,name
0,employee,1,Alice Johnson
1,employee,2,Bob Smith
2,employee,3,Carol White
3,employee,4,David Brown
4,employee,5,Eva Green
5,employee,6,Frank Moore
6,employee,7,Grace Lee
7,employee,8,Henry Clark
8,employee,9,Irene Lewis
9,employee,10,Jack Walker


**Why / traps:** Column names come from the **first** query.

## Q2. Customers active in 2023 or 2025  ·  🟢 easy

Show each `customer_id` that ordered in 2023 **or** in 2025, once.

*Expected result: 27 rows · columns: `customer_id`*

**How it works:** Customers of 2023 `UNION` customers of 2025: duplicates are removed, so each id appears once.

In [ ]:
%%bigquery
SELECT customer_id FROM shield.orders WHERE EXTRACT(YEAR FROM order_date) = 2023
UNION DISTINCT
SELECT customer_id FROM shield.orders WHERE EXTRACT(YEAR FROM order_date) = 2025

,customer_id
0,18
1,3
2,7
3,19
4,21
5,27
6,29
7,30
8,2
9,11


**Why / traps:** `UNION` removes duplicates (extra work: a sort or hash). Use `UNION ALL` when you know there are none, or when you want them.

## Q3. Customers active in both 2023 and 2025  ·  🟢 easy

Show each `customer_id` that ordered in 2023 **and** in 2025.

*Expected result: 4 rows · columns: `customer_id`*

**How it works:** `INTERSECT` keeps only ids that appear in **both** results.

In [ ]:
%%bigquery
SELECT customer_id FROM shield.orders WHERE EXTRACT(YEAR FROM order_date) = 2023
INTERSECT DISTINCT
SELECT customer_id FROM shield.orders WHERE EXTRACT(YEAR FROM order_date) = 2025

,customer_id
0,21
1,27
2,29
3,30


## Q4. Lost customers  ·  🟢 easy

Show each `customer_id` that ordered in 2024 but **not** in 2025.

*Expected result: 7 rows · columns: `customer_id`*

**How it works:** `EXCEPT` / `MINUS` keeps ids from the first result that are **not** in the second.

In [ ]:
%%bigquery
SELECT customer_id FROM shield.orders WHERE EXTRACT(YEAR FROM order_date) = 2024
EXCEPT DISTINCT
SELECT customer_id FROM shield.orders WHERE EXTRACT(YEAR FROM order_date) = 2025

,customer_id
0,1
1,13
2,18
3,19
4,20
5,6
6,8


**Why / traps:** Same as an anti join. EXCEPT also removes duplicates. Interview follow-up: "write it with NOT EXISTS".

## Q5. January users who did not come back in February  ·  🟡 medium

Show each `user_id` that logged in during January 2025 but **not** during February 2025.

*Expected result: 0 rows · columns: `user_id`*

**How it works:** Same as Q4 with login dates: January users minus February users = users who did not come back.

In [ ]:
%%bigquery
SELECT user_id FROM shield.user_logins
WHERE login_date >= DATE '2025-01-01' AND login_date < DATE '2025-02-01'
EXCEPT DISTINCT
SELECT user_id FROM shield.user_logins
WHERE login_date >= DATE '2025-02-01' AND login_date < DATE '2025-03-01'

,user_id


**Why / traps:** This is **churn** (users who stopped coming). Retention questions in topic 08 build on it.

## Q6. Status summary with a TOTAL row  ·  🟡 medium

Show `status` and `num_orders` (different orders) per status, then **one more row** `TOTAL` with the count of all different orders. Use UNION ALL.

*Expected result: 5 rows · columns: `status`, `num_orders`*

**How it works:** First SELECT = counts per status. Second SELECT = one row with the total. `UNION ALL` stacks them.

In [ ]:
%%bigquery
SELECT status, COUNT(DISTINCT order_id) AS num_orders
FROM shield.orders
GROUP BY status
UNION ALL
SELECT 'TOTAL', COUNT(DISTINCT order_id)
FROM shield.orders

,status,num_orders
0,cancelled,16
1,completed,70
2,pending,8
3,returned,10
4,TOTAL,104


**Why / traps:** Why not use ROLLUP? With COUNT(DISTINCT) both work; UNION ALL is easier to read. Here the total (104) equals the sum because each order has one status.

## Q7. Do two tables hold the same rows?  ·  🔴 hard

Is `orders` the same as `orders` **without duplicates**? Count rows that are in one but not the other, **in both directions**, keeping duplicates. Show `only_in_raw`, `only_in_clean`.

Hint: one row that appears twice in raw and once in clean counts once.

*Expected result: 1 row · columns: `only_in_raw`, `only_in_clean`*

**How it works:** `EXCEPT ALL` subtracts row by row (keeps duplicates). raw − clean leaves the 4 extra copies; clean − raw leaves nothing.

In [ ]:
%%bigquery
WITH raw AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY order_id, customer_id, order_date, amount, status) AS copy_no
             FROM shield.orders),
clean AS (SELECT *, 1 AS copy_no FROM (SELECT DISTINCT * FROM shield.orders))
SELECT
  (SELECT COUNT(*) FROM (SELECT * FROM raw   EXCEPT DISTINCT SELECT * FROM clean)) AS only_in_raw,
  (SELECT COUNT(*) FROM (SELECT * FROM clean EXCEPT DISTINCT SELECT * FROM raw))   AS only_in_clean

,only_in_raw,only_in_clean
0,4,0


**Why / traps:** `EXCEPT` (DISTINCT) would say "0 differences" and hide the 4 extra copies. Comparing tables both ways is how you test a pipeline: source vs target.

BigQuery has no `EXCEPT ALL`. The trick: number each copy with ROW_NUMBER, then plain EXCEPT DISTINCT sees copy 1 and copy 2 as different rows. A window with only PARTITION BY (no ORDER BY) is allowed for ROW_NUMBER here.

## Q8. Products: sold, never sold, all  ·  🟡 medium

Show `status` (`sold` / `never sold`) and `num_products`, using a UNION ALL of two queries.

*Expected result: 2 rows · columns: `status`, `num_products`*

**How it works:** Count sold products and never-sold products in two queries, label each, stack with UNION ALL.

In [ ]:
%%bigquery
SELECT 'sold' AS status, COUNT(*) AS num_products
FROM shield.products WHERE product_id IN (SELECT product_id FROM shield.order_items)
UNION ALL
SELECT 'never sold', COUNT(*)
FROM shield.products WHERE product_id NOT IN (SELECT product_id FROM shield.order_items)

,status,num_products
0,sold,18
1,never sold,2


**Why / traps:** `NOT IN` is safe **here** only because `order_items.product_id` has no NULLs. In real code prefer NOT EXISTS.